# Deutsch-Jozsa Algorithm Implementation

## 1. Executive Summary
The **Deutsch-Jozsa algorithm** provides a deterministic quantum speedup over classical algorithms. Given a black-box function $f: \{0,1\}^n \rightarrow \{0,1\}$, it determines whether $f$ is **constant** (outputs $0$ for all inputs, or $1$ for all inputs) or **balanced** (outputs $0$ for half the inputs, and $1$ for the other half).

* **Classical Complexity:** Requires up to $2^{n-1} + 1$ evaluations in the worst-case deterministic model.
* **Quantum Complexity:** Requires exactly **1 evaluation** of the oracle $U_f$.

### Physical Engine: Phase Kickback
By setting the target qubit to $\vert{}-\rangle = \frac{\vert{}0\rangle - \vert{}1\rangle}{\sqrt{2}}$, applying the oracle $U_f \vert{}x\rangle \vert{}-\rangle = (-1)^{f(x)} \vert{}x\rangle \vert{}-\rangle$ kicks back the function value $f(x)$ directly as a phase factor $(-1)^{f(x)}$ onto the input control state $\vert{}x\rangle$.

---

## 2. Environment Setup

In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

## 3. Oracle Construction
We define dynamic oracle gates $U_f$ operating on an $(n+1)$-qubit system ($n$ input qubits + $1$ target qubit):
* **Constant Oracle:** $f(x) = 1$. Applies an $X$ gate to the target qubit unconditionally.
* **Balanced Oracle:** $f(x) = x_0 \oplus x_1 \oplus \dots \oplus x_{n-1}$. Applies $CX$ (CNOT) gates targeting the helper qubit from each input qubit.

In [2]:
def build_dj_oracle(n: int, case: str) -> QuantumCircuit:
    """Generates a straightforward Deutsch-Jozsa oracle."""
    oracle = QuantumCircuit(n + 1)
    
    if case == "constant":
        # f(x) = 1 everywhere: flips target qubit unconditionally
        oracle.x(n)
    elif case == "balanced":
        # f(x) = 1 for half of input states via CNOTs
        for qubit in range(n):
            oracle.cx(qubit, n)
            
    return oracle.to_gate(label=f"Oracle ({case.capitalize()})")

## 4. Algorithm Circuit Design
The circuit runs through 5 distinct operational stages:
1. **State Initialization:** Drive target qubit to $\vert{}1\rangle$.
2. **Superposition:** Apply $H^{\otimes (n+1)}$ to transform inputs into an equal superposition and target to $\vert{}-\rangle$.
3. **Oracle Query:** Apply $U_f$ to trigger phase kickback.
4. **Interference Generation:** Apply $H^{\otimes n}$ to the input register to force constructive/destructive interference.
5. **Measurement:** Read out the $n$ input control qubits.

In [3]:
def build_deutsch_jozsa(n: int, case: str) -> QuantumCircuit:
    """Constructs the complete Deutsch-Jozsa circuit."""
    qc = QuantumCircuit(n + 1, n)
    
    # 1. Initialize target qubit to |1>
    qc.x(n)
    
    # 2. Apply Hadamard to all qubits (target qubit becomes |->)
    qc.h(range(n + 1))
    qc.barrier()
    
    # 3. Apply Oracle (Phase Kickback occurs here)
    oracle_gate = build_dj_oracle(n, case)
    qc.append(oracle_gate, range(n + 1))
    qc.barrier()
    
    # 4. Interference: Apply final Hadamards to input qubits
    qc.h(range(n))
    
    # 5. Measure input qubits
    qc.measure(range(n), range(n))
    return qc

## 5. Simulation & Verification
We execute both oracle paths on a 3-qubit input register ($n=3$) using `AerSimulator`.

In [6]:
# --- Execution ---
n = 3
sim = AerSimulator()

for case in ["constant", "balanced"]:
    circuit = build_deutsch_jozsa(n, case)
    
    # Transpile circuit so the simulator can read the custom oracle gate
    compiled_circuit = transpile(circuit, sim)
    
    counts = sim.run(compiled_circuit, shots=1024).result().get_counts()
    print(f"{case.capitalize()} Oracle Measurement:", counts)

Constant Oracle Measurement: {'000': 1024}
Balanced Oracle Measurement: {'111': 1024}


## 6. Mathematical Results Analysis

| Oracle Type | Measured Output | Probability | Physical Interpretation |
| :--- | :--- | :--- | :--- |
| **Constant** | `000` | **100%** | Uniform negative phase $(-1)^1$ results in total **constructive interference** back to $\vert 000 \rangle$. |
| **Balanced** | `111` | **100%** | Alternating phase signs $(-1)^{x_0+x_1+x_2}$ cause complete **destructive interference** at $\vert 000 \rangle$, shifting all probability amplitude into non-zero basis states. |